# 🌍 Notebook 12: Coleta Multi-Fonte de Notícias (2018-2025)

**Objetivo:** Construir base histórica robusta de notícias em português sobre Ibovespa/mercado financeiro brasileiro.

**Estratégia de Coleta:**

1. **GDELT** (fonte primária histórica)
   - Cobertura: 2015-presente (usaremos 2018-2025)
   - Limitação: Não retorna texto completo (apenas título + metadados)
   - Vantagem: Gratuito, sem limite de histórico, atualização contínua
   - Taxa: ~50 req/min (com delay de 1.5s entre chamadas)

2. **NewsAPI** (fonte complementar recente)
   - Cobertura: Últimos 30 dias (plano FREE)
   - Vantagem: Alta qualidade, descrição + snippet de conteúdo
   - Limitação: Histórico limitado a 30 dias no plano gratuito
   - Taxa: 100 req/dia

**Schema Unificado:**
- `date`: datetime normalizado (meia-noite BRT)
- `source`: nome do veículo/domínio
- `title`: título da notícia
- `url`: link do artigo
- `text_full`: texto disponível (título + descrição/conteúdo)

**Resultado Final:**
- `data_raw/news_multisource.parquet`: Base bruta consolidada
- Cobertura esperada: ~2018-2025 (dentro das limitações das APIs)

---

In [1]:
# 1. Setup de caminhos e constantes
import os
import sys
import subprocess
from datetime import datetime, timedelta
from pathlib import Path

# Adiciona src ao path para imports
sys.path.insert(0, str(Path(".").absolute().parent))

from src.io import paths
from src.config import loader as cfg
from src.config.constants import START_DATE, END_DATE, START_DATE_STR, END_DATE_STR

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "12_data_collection_multisource"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("="*80)
print(f"NOTEBOOK: {NB_NAME}")
print(f"EXECUÇÃO: {STAMP}")
print("="*80)
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"RAW_PATH: {RAW_PATH}")
print(f"\n📅 PERÍODO ALVO (constants.py):")
print(f"   START_DATE: {START_DATE} ({START_DATE_STR})")
print(f"   END_DATE: {END_DATE} ({END_DATE_STR})")
print(f"   Dias teóricos: {(END_DATE - START_DATE).days + 1}")
print("="*80)

NOTEBOOK: 12_data_collection_multisource
EXECUÇÃO: 20251119_160645

BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
RAW_PATH: C:\Users\ander\OneDrive\TCC_USP\data_raw

📅 PERÍODO ALVO (constants.py):
   START_DATE: 2018-01-02 (2018-01-02)
   END_DATE: 2025-12-31 (2025-12-31)
   Dias teóricos: 2921


In [2]:
# 2. Importações e carregamento dos coletores
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Import dos coletores customizados
from src.utils.gdelt_collector import GDELTCollector, collect_gdelt_historical
from src.utils.newsapi_collector import NewsAPICollector, collect_newsapi_recent

print("✅ Coletores importados com sucesso")
print("   - GDELTCollector: Coleta histórica 2015-presente")
print("   - NewsAPICollector: Coleta últimos 30 dias (plano FREE)")

✅ Coletores importados com sucesso
   - GDELTCollector: Coleta histórica 2015-presente
   - NewsAPICollector: Coleta últimos 30 dias (plano FREE)


In [5]:
# 3. Configuração da coleta

# Chave NewsAPI (encontrada nos runs anteriores - VÁLIDA)
NEWSAPI_KEY = "3d606b02f24447849f49b3e8c3628f46"

# Modo de execução
# "FULL": coleta histórica completa GDELT (2018-2025) + NewsAPI recente (pode levar horas)
# "RECENT": apenas NewsAPI últimos 30 dias (execução rápida ~2min) ← RECOMENDADO INICIALMENTE
# "TEST": teste com 7 dias GDELT + 7 dias NewsAPI (validação rápida)

COLLECT_MODE = "RECENT"  # NewsAPI 30 dias - funcional e rápido

print("="*80)
print(f"MODO DE COLETA: {COLLECT_MODE}")
print("="*80)

if COLLECT_MODE == "FULL":
    print("⚠️ COLETA COMPLETA HISTÓRICA")
    print(f"   Período GDELT: {START_DATE_STR} → {END_DATE_STR}")
    print(f"   Período NewsAPI: Últimos 30 dias")
    print(f"   Tempo estimado: 4-8 horas (depende da API)")
    print(f"   Resultado: ~50.000-200.000 artigos (estimativa)")
    print(f"\n⚠️ ATENÇÃO: GDELT requer ajuste no parser de datas")
elif COLLECT_MODE == "RECENT":
    print("📰 COLETA RECENTE (NewsAPI)")
    print(f"   Período: Últimos 30 dias")
    print(f"   Tempo estimado: 2-5 minutos")
    print(f"   Cobertura: ~500-2000 artigos esperados")
    print(f"\n✅ RECOMENDADO para validação inicial da pipeline")
elif COLLECT_MODE == "TEST":
    print("🧪 MODO TESTE")
    print(f"   Período: Últimos 7 dias (GDELT + NewsAPI)")
    print(f"   Tempo estimado: 1-2 minutos")
    print(f"   ⚠️ GDELT pode precisar de ajustes")
    
print("="*80)

MODO DE COLETA: RECENT
📰 COLETA RECENTE (NewsAPI)
   Período: Últimos 30 dias
   Tempo estimado: 2-5 minutos
   Cobertura: ~500-2000 artigos esperados

✅ RECOMENDADO para validação inicial da pipeline


In [6]:
# 4. Coleta via GDELT (histórico)

print("\n" + "="*80)
print("ETAPA 1: COLETA GDELT")
print("="*80 + "\n")

if COLLECT_MODE == "FULL":
    # Coleta histórica completa: 2018-2025
    print("🌍 Iniciando coleta GDELT histórica...")
    print(f"⏱️ Isso pode levar 4-8 horas para {(END_DATE - START_DATE).days} dias")
    print("💾 Checkpoints serão salvos a cada 30 dias\n")
    
    df_gdelt = collect_gdelt_historical(
        start_date=START_DATE_STR,
        end_date=END_DATE_STR,
        output_path=Path(RAW_PATH) / "gdelt_historical.parquet",
        checkpoint_interval=30,
    )
    
elif COLLECT_MODE == "TEST":
    # Teste: últimos 7 dias
    print("🧪 Teste GDELT: últimos 7 dias")
    collector = GDELTCollector(rate_limit_delay=0.5)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=7)
    
    df_gdelt = collector.collect_by_date_range(
        start_date=start_date,
        end_date=end_date,
        query="(Ibovespa OR Bovespa OR bolsa OR ações OR mercado) sourcelang:por",
        max_records=250,
    )
    
else:  # RECENT mode
    print("⏭️ Modo RECENT: pulando coleta GDELT (use FULL para histórico completo)")
    df_gdelt = pd.DataFrame()

# Resumo GDELT
if not df_gdelt.empty:
    print("\n📊 Resumo GDELT:")
    print(f"   Total: {len(df_gdelt):,} artigos")
    print(f"   Período: {df_gdelt['date'].min()} → {df_gdelt['date'].max()}")
    print(f"   Dias distintos: {df_gdelt['date'].nunique():,}")
    print(f"   Fontes únicas: {df_gdelt['source'].nunique():,}")
    display(df_gdelt.head(3))
else:
    print("⚠️ GDELT: Nenhum dado coletado (modo RECENT ou erro)")


ETAPA 1: COLETA GDELT

⏭️ Modo RECENT: pulando coleta GDELT (use FULL para histórico completo)
⚠️ GDELT: Nenhum dado coletado (modo RECENT ou erro)


In [7]:
# 5. Coleta via NewsAPI (recente)

print("\n" + "="*80)
print("ETAPA 2: COLETA NewsAPI")
print("="*80 + "\n")

if COLLECT_MODE in ["FULL", "RECENT"]:
    days_to_collect = 30  # Máximo do plano FREE
elif COLLECT_MODE == "TEST":
    days_to_collect = 7  # Teste
else:
    days_to_collect = 0

if days_to_collect > 0:
    print(f"📰 Coletando NewsAPI: últimos {days_to_collect} dias")
    
    collector_newsapi = NewsAPICollector(
        api_key=NEWSAPI_KEY,
        rate_limit_delay=1.0,
    )
    
    df_newsapi = collector_newsapi.collect_recent(days=days_to_collect)
    
    # Resumo NewsAPI
    if not df_newsapi.empty:
        print("\n📊 Resumo NewsAPI:")
        print(f"   Total: {len(df_newsapi):,} artigos")
        print(f"   Período: {df_newsapi['date'].min()} → {df_newsapi['date'].max()}")
        print(f"   Dias distintos: {df_newsapi['date'].nunique():,}")
        print(f"   Fontes únicas: {df_newsapi['source'].nunique():,}")
        display(df_newsapi.head(3))
    else:
        print("⚠️ NewsAPI: Nenhum dado retornado")
else:
    print("⏭️ NewsAPI: pulado (nenhum dia configurado)")
    df_newsapi = pd.DataFrame()


ETAPA 2: COLETA NewsAPI

📰 Coletando NewsAPI: últimos 30 dias
📰 NewsAPI: Coletando 2025-10-20 → 2025-11-19
  ✅ Página 1: 99 artigos
✅ Total: 98 artigos
   Período: 2025-11-18 00:00:00+00:00 → 2025-11-18 00:00:00+00:00
   Fontes: 16

📊 Resumo NewsAPI:
   Total: 98 artigos
   Período: 2025-11-18 00:00:00+00:00 → 2025-11-18 00:00:00+00:00
   Dias distintos: 1
   Fontes únicas: 16
  ✅ Página 1: 99 artigos
✅ Total: 98 artigos
   Período: 2025-11-18 00:00:00+00:00 → 2025-11-18 00:00:00+00:00
   Fontes: 16

📊 Resumo NewsAPI:
   Total: 98 artigos
   Período: 2025-11-18 00:00:00+00:00 → 2025-11-18 00:00:00+00:00
   Dias distintos: 1
   Fontes únicas: 16


,date,source,title,url,text_full
0,2025-11-18 00:00:00+00:00,Abril.com.br,Por que a prisão de dono do Banco Master cai c...,https://veja.abril.com.br/politica/por-que-a-p...,Operação da PF atinge banqueiro Daniel Vorcaro...
1,2025-11-18 00:00:00+00:00,Metropoles.com,Justiça autoriza bloqueio de bens do BRB e do ...,https://www.metropoles.com/colunas/grande-angu...,"Além das instituições financeiras, foram alvos..."
2,2025-11-18 00:00:00+00:00,Abril.com.br,Quem tem fundos de investimento no Banco Maste...,https://veja.abril.com.br/economia/quem-tem-fu...,Operação Compliance Zero foi deflagrada hoje p...


In [ ]:
# 6. Consolidação e salvamento final

print("\n" + "="*80)
print("ETAPA 3: CONSOLIDAÇÃO E SALVAMENTO")
print("="*80 + "\n")

# Consolidar ambas as fontes
dataframes = []

if not df_gdelt.empty:
    print(f"✅ GDELT: {len(df_gdelt):,} artigos")
    dataframes.append(df_gdelt)

if not df_newsapi.empty:
    print(f"✅ NewsAPI: {len(df_newsapi):,} artigos")
    dataframes.append(df_newsapi)

if not dataframes:
    print("❌ ERRO: Nenhuma fonte retornou dados!")
    print("   Verifique:")
    print("   - Chave NewsAPI válida")
    print("   - Conexão com internet")
    print("   - Rate limits das APIs")
    raise ValueError("Coleta falhou - nenhum dado obtido")

# Concatenar e deduplic ar
df_consolidated = pd.concat(dataframes, ignore_index=True)

print(f"\n📊 Antes da deduplicação: {len(df_consolidated):,} artigos")

# Deduplicação por URL e título
df_consolidated = df_consolidated.drop_duplicates(subset=["url", "title"])
df_consolidated = df_consolidated.sort_values("date").reset_index(drop=True)

print(f"📊 Após deduplicação: {len(df_consolidated):,} artigos")

# Validação de qualidade
df_consolidated = df_consolidated.dropna(subset=["date", "title"])
df_consolidated = df_consolidated[df_consolidated["title"].str.len() >= 20]

print(f"📊 Após filtros de qualidade: {len(df_consolidated):,} artigos\n")

# Estatísticas finais
print("="*80)
print("ESTATÍSTICAS FINAIS")
print("="*80)
print(f"Total de artigos: {len(df_consolidated):,}")
print(f"Período coberto: {df_consolidated['date'].min().date()} → {df_consolidated['date'].max().date()}")
print(f"Dias distintos: {df_consolidated['date'].nunique():,}")
print(f"Fontes únicas: {df_consolidated['source'].nunique():,}")
print(f"Média diária: {len(df_consolidated) / df_consolidated['date'].nunique():.1f} artigos/dia")
print("="*80 + "\n")

# Top 10 fontes
print("📰 Top 10 fontes mais frequentes:")
print(df_consolidated["source"].value_counts().head(10))

# Amostra
print("\n📄 Amostra dos dados:")
display(df_consolidated.sample(min(5, len(df_consolidated))))

# Salvamento em data_raw (arquivo bruto)
output_file = Path(RAW_PATH) / "news_multisource.parquet"
df_consolidated.to_parquet(output_file, index=False)

print(f"\n💾 Arquivo salvo em: {output_file}")
print(f"   Tamanho: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

# Backup CSV (para inspeção manual)
output_csv = Path(RAW_PATH) / "news_multisource.csv"
df_consolidated.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"   Backup CSV: {output_csv}")

print("\n✅ COLETA CONCLUÍDA COM SUCESSO!")
print("="*80)

⚠ Sem dados da API — carregando fallback: /content/drive/MyDrive/TCC_USP/data_raw/noticias_real.csv
Após normalização: (81, 3)


,day,fonte,texto
0,2025-09-27,IGN,Esqueceu ou decidiu não comprar a Pena? O game...
1,2025-09-27,Sapo.pt,O Google Fotos poderá receber uma funcionalida...
2,2025-09-27,Observador.pt,Fogo deflagrou por volta das 11:50 deste sábad...
3,2025-09-27,Abril.com.br,Vale conferir as dicas preparadas especialment...
4,2025-09-27,Metropoles.com,Cena de Vale Tudo foi detonada nas redes socia...


✅ Arquivos salvos:
/content/drive/MyDrive/TCC_USP/data_processed/news_multisource_clean.csv
/content/drive/MyDrive/TCC_USP/data_processed/news_multisource_clean.parquet
Período: 2025-09-27 00:00:00 → 2025-09-27 00:00:00
Fontes únicas: 21
Total de notícias: 81


,day,fonte,texto
23,2025-09-27,Metropoles.com,Tudo começa pelo mínimo de doação de 200 mil d...
53,2025-09-27,Observador.pt,Ministério do Ambiente explica que o país pode...
14,2025-09-27,Terra.com.br,"Neste sábado (27), no Riyadh Air Metropolitano..."


Feito ✅
